In [ ]:
# ==============================
# GAME METADATA
# ==============================

def get_game_name(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        if data[str(app_id)]["success"]:
            return data[str(app_id)]["data"]["name"]
    except:
        pass
    return f"Unknown Game ({app_id})"


# ==============================
# PLAYER STATS
# ==============================

def get_player_stats(app_id, game_name):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        current_players = data["response"]["player_count"]
    except:
        current_players = "N/A"

    print(f"\n=== PLAYER SNAPSHOT: {game_name} ===")
    if current_players != "N/A":
        print(f"  Current Players:  {current_players:,}")
    else:
        print(f"  Current Players:  N/A")

# ==============================
# TEXT CLEANING
# ==============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text


# ==============================
# DATA COLLECTION
# ==============================

def fetch_reviews(app_id, source_name, num_reviews=1000):
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    params = {
        "json": 1,
        "filter": "all",
        "language": "english",
        "day_range": 3650,
        "review_type": "all",
        "purchase_type": "all",
        "num_per_page": 100
    }

    reviews = []
    cursor = "*"

    with tqdm(total=num_reviews, desc=f"Fetching {source_name}") as pbar:
        while len(reviews) < num_reviews:
            params["cursor"] = cursor

            try:
                r = requests.get(url, params=params)
                data = r.json()
            except:
                time.sleep(2)
                continue

            if not data.get("reviews"):
                break

            for review in data["reviews"]:
                author = review.get("author", {})

                reviews.append({
                    "steamid": str(author.get("steamid")),
                    "review_text": review.get("review"),
                    "recommended": review.get("voted_up"),
                    "source": source_name
                })

                pbar.update(1)

                if len(reviews) >= num_reviews:
                    break

            cursor = data.get("cursor")
            time.sleep(1)

    df = pd.DataFrame(reviews)
    df["review_text"] = df["review_text"].apply(clean_text)

    return df

In [ ]:
# ==============================
# SENTIMENT MODEL
# ==============================

def analyze_sentiment(df, label):
    X = df["review_text"].fillna("")
    y = df["recommended"].map({True: "Recommend", False: "Not Recommend"})

    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    X_vec = vectorizer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vec, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"\n=== SENTIMENT MODEL ({label}) ===")
    print(classification_report(y_test, preds))

    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]

    n = 15
    top_negative_idx = coef.argsort()[:n]
    top_positive_idx = coef.argsort()[-n:][::-1]

    print(f"\nTop words driving NOT RECOMMEND:")
    for idx in top_negative_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    print(f"\nTop words driving RECOMMEND:")
    for idx in top_positive_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    return model, vectorizer


# ==============================
# BERTOPIC
# ==============================

def analyze_topics_bertopic(df):
    print("\n=== TOPIC MODEL (BERTopic) ===")

    docs = df["review_text"].dropna().tolist()

    steam_stopwords = [
        "game", "play", "played", "playing",
        "like", "just", "get", "one", "will"
    ]

    vectorizer_model = CountVectorizer(
        stop_words=list(ENGLISH_STOP_WORDS) + steam_stopwords,
        ngram_range=(1, 2),
        min_df=5
    )

    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(docs)

    topic_info = topic_model.get_topic_info()

    print("\nTop Topics Found:")
    print(topic_info.head(15))

    print("\nSample Clean Topic Words:\n")
    for topic_id in topic_info["Topic"].head(10):
        if topic_id == -1:
            continue

        words = topic_model.get_topic(topic_id)
        words = [w[0] for w in words[:10]]
        print(f"Topic {topic_id}: {', '.join(words)}")

    return topic_model


# ==============================
# TOPIC x SENTIMENT BREAKDOWN
# ==============================

def topic_sentiment_breakdown(df, topic_model, sentiment_model, vectorizer):
    docs = df["review_text"].dropna().reset_index(drop=True)
    labels = df["recommended"].dropna().reset_index(drop=True)

    topics, _ = topic_model.transform(docs.tolist())

    X_vec = vectorizer.transform(docs)
    preds = sentiment_model.predict(X_vec)

    results = pd.DataFrame({
        "topic": topics,
        "predicted_sentiment": preds,
        "actual": labels.map({True: "Recommend", False: "Not Recommend"})
    })

    rows = []
    for topic_id, group in results[results["topic"] != -1].groupby("topic"):
        total = len(group)
        neg_pct = (group["predicted_sentiment"] == "Not Recommend").mean() * 100
        pos_pct = 100 - neg_pct
        words = topic_model.get_topic(topic_id)
        label = ", ".join([w[0] for w in words[:3]])
        rows.append((topic_id, label, total, pos_pct, neg_pct))

    negative_side = sorted([r for r in rows if r[4] > 50], key=lambda x: x[4], reverse=True)
    positive_side = sorted([r for r in rows if r[4] <= 50], key=lambda x: x[3], reverse=True)

    print("\n=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===")

    print("\n--- MOST NEGATIVE ---")
    for r in negative_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👎 {r[4]:.1f}%  |  👍 {r[3]:.1f}%")

    print("\n--- MOST POSITIVE ---")
    for r in positive_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👍 {r[3]:.1f}%  |  👎 {r[4]:.1f}%")

    return results

In [ ]:
# ==============================
# FULL PIPELINE
# ==============================

def analyze_full_game(base_app_id, dlc_app_id=None, num_reviews=4000):
    game_name = get_game_name(base_app_id)
    print(f"\n===== BASE GAME: {game_name} (ID: {base_app_id}) =====")

    # Player snapshot at the top
    get_player_stats(base_app_id, game_name)

    base_df = fetch_reviews(base_app_id, "base", num_reviews)
    model, vectorizer = analyze_sentiment(base_df, "base")
    topic_model = analyze_topics_bertopic(base_df)
    breakdown = topic_sentiment_breakdown(base_df, topic_model, model, vectorizer)

    base_positive_rate = base_df["recommended"].mean()
    print("\n=== OVERALL BASE GAME SENTIMENT ===")
    print(f"Positive Rate: {base_positive_rate:.2f}")
    print(f"Negative Rate: {1 - base_positive_rate:.2f}")

    if dlc_app_id:
        dlc_name = get_game_name(dlc_app_id)
        print(f"\n===== DLC: {dlc_name} (ID: {dlc_app_id}) =====")

        # Player snapshot for DLC
        get_player_stats(dlc_app_id, dlc_name)

        dlc_df = fetch_reviews(dlc_app_id, "dlc", num_reviews // 2)
        analyze_sentiment(dlc_df, "dlc")
        analyze_topics_bertopic(dlc_df)
        topic_sentiment_breakdown(dlc_df, topic_model, model, vectorizer)

        dlc_positive_rate = dlc_df["recommended"].mean()
        print("\n=== OVERALL DLC SENTIMENT ===")
        print(f"Positive Rate: {dlc_positive_rate:.2f}")
        print(f"Negative Rate: {1 - dlc_positive_rate:.2f}")

        print("\n=== BASE vs DLC COMPARISON ===")
        print(f"Base positive rate:  {base_positive_rate:.2f}")
        print(f"DLC positive rate:   {dlc_positive_rate:.2f}")
        diff = dlc_positive_rate - base_positive_rate
        direction = "better" if diff > 0 else "worse"
        print(f"DLC was received {abs(diff)*100:.1f}% {direction} than the base game")

    return base_df, topic_model, model, vectorizer, breakdown

# **Tests of Recent Popular Games with App ID (AID)**

### **Elden Ring**

In [ ]:
# ===================================================================================
  # Elden Ring                      ->     AID: 1245620 (VERY Popular Game)
  # ARC Raiders                     ->     AID: 1808500 (Currently Most Played Game)

  # Call of Duty: Black Ops 7       ->     AID: 1938090 (NEGATIVE Reviews)
  # Shadow Of Doubt                 ->     AID: 1938090 (Niche Smaller Game)

  # Ninja Gaiden 4                  ->     AID: 2627260
  #  ''     ''   '' DLC             ->     AID: 4191490 (Test with DLC Added)

  # Marvel Rivals                   ->     AID: 2767030 (Popular Live Service Game)
  # HellDivers 2                    ->     AID: 553850  (Live Service Game)
# ===================================================================================


# ==============================
# BEGIN TESTS
# ==============================

# Elden Ring
data = analyze_full_game(1245620)


===== BASE GAME: ELDEN RING (ID: 1245620) =====

=== PLAYER SNAPSHOT: ELDEN RING ===
  Current Players:  38,227


Fetching base: 100%|██████████| 4000/4000 [01:06<00:00, 60.23it/s]
2026-05-08 14:18:07,708 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.31      0.24      0.27        17
    Recommend       0.98      0.99      0.99       783

     accuracy                           0.97       800
    macro avg       0.65      0.61      0.63       800
 weighted avg       0.97      0.97      0.97       800


Top words driving NOT RECOMMEND:
  terrible                  coef: -3.6752
  fix                       coef: -3.3972
  save                      coef: -3.1139
  sekiro                    coef: -2.8899
  don                       coef: -2.6816
  fans                      coef: -2.5774
  insan                     coef: -2.5391
  launch                    coef: -2.5370
  instead                   coef: -2.3917
  taken                     coef: -2.3235
  away                      coef: -2.2996
  level                     coef: -2.2721
  issues                    coef: -2.2083
  garbage                   coef: -2.1921
  doesn      

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

2026-05-08 14:22:08,407 - BERTopic - Embedding - Completed ✓
2026-05-08 14:22:08,410 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:22:58,462 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:22:58,463 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 14:22:58,649 - BERTopic - Cluster - Completed ✓
2026-05-08 14:22:58,656 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 14:22:59,498 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                          Name  \
0      -1   1268                        -1_world_fun_time_good   
1       0    806                 0_elden_ring_elden ring_world   
2       1    247                        1_bosses_boss_beat_fun   
3       2    203                2_souls_souls games_games_best   
4       3    100                            3_ps_pc_xbox_hours   
5       4     95            4_dark souls_dark_souls_open world   
6       5     92           5_open world_open_world_world games   
7       6     87                6_fun_hard_difficult_challenge   
8       7     78                         7_fps_pc_runs_support   
9       8     71  8_fromsoft_fromsoftware_games_fromsoft games   
10      9     68                    9_rage_hate_suffering_love   
11     10     67                       10_seamless_mod_coop_op   
12     11     60              11_recommend_sure_make sure_best   
13     12     58                           12_dlc_buy_ng_

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 14:25:50,644 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 14:25:50,660 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:25:50,661 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 14:25:50,820 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---

--- MOST POSITIVE ---
  Topic 4 [dark souls, dark, souls]
    Reviews: 95  |  👍 100.0%  |  👎 0.0%
  Topic 5 [open world, open, world]
    Reviews: 92  |  👍 100.0%  |  👎 0.0%
  Topic 6 [fun, hard, difficult]
    Reviews: 87  |  👍 100.0%  |  👎 0.0%
  Topic 9 [rage, hate, suffering]
    Reviews: 68  |  👍 100.0%  |  👎 0.0%
  Topic 14 [best, greatest, games]
    Reviews: 41  |  👍 100.0%  |  👎 0.0%
  Topic 15 [lore, amazing, story]
    Reviews: 40  |  👍 100.0%  |  👎 0.0%
  Topic 17 [souls, bosses, souls games]
    Reviews: 35  |  👍 100.0%  |  👎 0.0%
  Topic 19 [replay, best, time]
    Reviews: 33  |  👍 100.0%  |  👎 0.0%
  Topic 20 [story, graphics, great]
    Reviews: 32  |  👍 100.0%  |  👎 0.0%
  Topic 21 [malenia, sex, rot]
    Reviews: 31  |  👍 100.0%  |  👎 0.0%
  Topic 23 [peak, games peak, beautiful visuals]
    Reviews: 27  |  👍 100.0%  |  👎 0.0%
  Topic 24 [hours, surface, hours say]
    Reviews: 25  |  👍 100.0%  |  👎 0.0

### **Arc Raiders**

In [ ]:
# ARC Raiders
data = analyze_full_game(1808500)


===== BASE GAME: ARC Raiders (ID: 1808500) =====

=== PLAYER SNAPSHOT: ARC Raiders ===
  Current Players:  52,681


Fetching base: 100%|██████████| 4000/4000 [01:02<00:00, 63.52it/s]
2026-05-08 14:26:55,035 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.87      0.90      0.89       424
    Recommend       0.89      0.85      0.87       376

     accuracy                           0.88       800
    macro avg       0.88      0.88      0.88       800
 weighted avg       0.88      0.88      0.88       800


Top words driving NOT RECOMMEND:
  just                      coef: -3.7386
  boring                    coef: -2.9225
  cheaters                  coef: -2.7576
  toxic                     coef: -2.5474
  pvp                       coef: -2.4544
  killed                    coef: -2.3846
  ruined                    coef: -2.1742
  players                   coef: -2.1304
  worst                     coef: -2.0905
  trash                     coef: -1.9493
  getting                   coef: -1.8400
  kill                      coef: -1.8314
  banned                    coef: -1.7498
  worse                     coef: -1.7071
  camping    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 14:31:04,904 - BERTopic - Embedding - Completed ✓
2026-05-08 14:31:04,906 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:31:35,013 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:31:35,015 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 14:31:35,198 - BERTopic - Cluster - Completed ✓
2026-05-08 14:31:35,205 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 14:31:36,124 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                              Name  \
0      -1   1530                         -1_pvp_people_players_fun   
1       0    677                 0_raiders_arc_arc raiders_players   
2       1    190  1_extraction_shooter_extraction shooter_shooters   
3       2    177                           2_pvp_pve_mode_pve mode   
4       3    148       3_extraction_extraction shooter_shooter_pvp   
5       4    131                          4_pvp_players_pve_people   
6       5     93                                5_rats_rat_pve_pvp   
7       6     88             6_tarkov_shooter_escape tarkov_escape   
8       7     87                          7_fun_friends_solo_great   
9       8     65                       8_solo_friendly_solos_trios   
10      9     65                 9_sound_design_sound design_audio   
11     10     55                     10_cheaters_devs_cheat_cheats   
12     11     54                       11_banned_support_ban_cheat   
1

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 14:35:33,500 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 14:35:33,516 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:35:33,517 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 14:35:33,658 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 16 [expedition, million, skill points]
    Reviews: 33  |  👎 100.0%  |  👍 0.0%
  Topic 21 [players, fix, pvp]
    Reviews: 27  |  👎 100.0%  |  👍 0.0%
  Topic 22 [tier, free, guns]
    Reviews: 26  |  👎 100.0%  |  👍 0.0%
  Topic 32 [cheaters, cheating, streamers]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 10 [cheaters, devs, cheat]
    Reviews: 55  |  👎 98.2%  |  👍 1.8%
  Topic 11 [banned, support, ban]
    Reviews: 54  |  👎 98.1%  |  👍 1.9%
  Topic 4 [pvp, players, pve]
    Reviews: 131  |  👎 96.9%  |  👍 3.1%
  Topic 25 [late, spawn, mins]
    Reviews: 25  |  👎 96.0%  |  👍 4.0%
  Topic 29 [crashes, restart, error]
    Reviews: 21  |  👎 95.2%  |  👍 4.8%
  Topic 13 [durability, nerf, map]
    Reviews: 40  |  👎 95.0%  |  👍 5.0%
  Topic 5 [rats, rat, pve]
    Reviews: 93  |  👎 90.3%  |  👍 9.7%
  Topic 20 [items, inventory, gear]
    Reviews: 29  |  👎 86.2%  |  👍 13.8%
  Topic 15 [toxic, passive, people]
    Review

### **Call of Duty: Black Ops 7**

In [ ]:
# COD Black Ops 7 (REVIEWED BOMBED)
data = analyze_full_game(1938090)


===== BASE GAME: Call of Duty® (ID: 1938090) =====

=== PLAYER SNAPSHOT: Call of Duty® ===
  Current Players:  29,321


Fetching base: 100%|██████████| 4000/4000 [01:01<00:00, 65.05it/s]
2026-05-08 14:36:36,459 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.95      0.90      0.92       673
    Recommend       0.57      0.74      0.65       127

     accuracy                           0.87       800
    macro avg       0.76      0.82      0.78       800
 weighted avg       0.89      0.87      0.88       800


Top words driving NOT RECOMMEND:
  worst                     coef: -2.7507
  money                     coef: -2.4901
  terrible                  coef: -2.4260
  trash                     coef: -2.3054
  just                      coef: -2.3035
  worse                     coef: -2.1250
  anymore                   coef: -2.1027
  activision                coef: -2.0957
  horrible                  coef: -2.0805
  battlefield               coef: -2.0111
  update                    coef: -1.9628
  waste                     coef: -1.9259
  making                    coef: -1.8778
  garbage                   coef: -1.8480
  slop       

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 14:39:01,547 - BERTopic - Embedding - Completed ✓
2026-05-08 14:39:01,548 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:39:32,711 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:39:32,713 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 14:39:32,872 - BERTopic - Cluster - Completed ✓
2026-05-08 14:39:32,878 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 14:39:33,331 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                           Name  \
0      -1   1427                           -1_cod_bo_games_time   
1       0    280                      0_fun_good_recommend_love   
2       1    256                  1_gb_download_install_warzone   
3       2    180                    2_cod_worst cod_worst_games   
4       3    162                    3_black ops_ops_black_feels   
5       4    135            4_zombies_multiplayer_campaign_mode   
6       5    125                    5_trash_money_free_terrible   
7       6    111                    6_duty_year_franchise_games   
8       7    107              7_warzone_royale_battle royale_bo   
9       8     84               8_cheaters_cheat_anti cheat_anti   
10      9     75                  9_tpm_boot_secure_secure boot   
11     10     68                   10_banned_ban_shadow_account   
12     11     60                         11_bo_bo bo_year_worse   
13     12     58  12_crashes_crashing_keeps

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 14:41:55,290 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 14:41:55,304 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:41:55,305 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 14:41:55,441 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 10 [banned, ban, shadow]
    Reviews: 68  |  👎 100.0%  |  👍 0.0%
  Topic 20 [account, phone number, phone]
    Reviews: 34  |  👎 100.0%  |  👍 0.0%
  Topic 25 [activision, year don, buy new]
    Reviews: 29  |  👎 100.0%  |  👍 0.0%
  Topic 27 [refund, steam, hours]
    Reviews: 27  |  👎 100.0%  |  👍 0.0%
  Topic 30 [shaders, restart, restart restart]
    Reviews: 25  |  👎 100.0%  |  👍 0.0%
  Topic 33 [bios, bios update, wont]
    Reviews: 21  |  👎 100.0%  |  👍 0.0%
  Topic 35 [launcher, worst, retarded]
    Reviews: 19  |  👎 100.0%  |  👍 0.0%
  Topic 36 [skins, maps, dollars]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 38 [packet, packet burst, burst]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 39 [slop, ai slop, ai]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 43 [activision, cheaters, blatant]
    Reviews: 13  |  👎 100.0%  |  👍 0.0%
  Topic 9 [tpm, boot, secure]
    Reviews: 75  |  👎 98.7%  |  👍 1.3%
  T

### **Shadow Of Doubt**

In [ ]:
# Shadow Of Doubt
data = analyze_full_game(986130)


===== BASE GAME: Shadows of Doubt (ID: 986130) =====

=== PLAYER SNAPSHOT: Shadows of Doubt ===
  Current Players:  189


Fetching base: 100%|██████████| 4000/4000 [01:04<00:00, 61.97it/s]
2026-05-08 14:43:01,488 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.85      0.91      0.88       249
    Recommend       0.96      0.93      0.94       551

     accuracy                           0.92       800
    macro avg       0.90      0.92      0.91       800
 weighted avg       0.92      0.92      0.92       800


Top words driving NOT RECOMMEND:
  just                      coef: -2.9661
  idea                      coef: -2.8305
  minutes                   coef: -2.5579
  unfinished                coef: -2.5185
  boring                    coef: -2.4229
  literally                 coef: -2.3739
  update                    coef: -2.3451
  buggy                     coef: -2.3289
  state                     coef: -2.3178
  performance               coef: -2.2689
  fix                       coef: -2.2232
  bad                       coef: -2.2101
  finished                  coef: -2.1426
  concept                   coef: -2.0993
  early      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 14:45:42,912 - BERTopic - Embedding - Completed ✓
2026-05-08 14:45:42,913 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:46:14,268 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:46:14,270 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 14:46:14,516 - BERTopic - Cluster - Completed ✓
2026-05-08 14:46:14,521 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 14:46:15,064 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                           Name  \
0      -1    377                 -1_building_keys_case_actually   
1       0     52                 0_fun fun_fun_challenging_hard   
2       1     50              1_tutorial_progress_logic_forward   
3       2     49                 2_price_sadly_execution_hollow   
4       3     48                                 3_thing_want__   
5       4     47        4_games_definitely recommend_enjoy_peak   
6       5     46                 5_ask_gov_environment_database   
7       6     46                 6_bro_fun love_great great_gem   
8       7     45             7_player_simulation_doubt_murderer   
9       8     44                     8_detective_man_im_pretend   
10      9     44                      9_buggy_soft_reload_state   
11     10     43                  10_overall_easier_glitchy_bad   
12     11     40  11_mechanics_progression_similar_murder cases   
13     12     40                    12_grea

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 14:48:53,985 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 14:48:54,001 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:48:54,002 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 14:48:54,130 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 141 [returned, runs, great concept]
    Reviews: 12  |  👎 100.0%  |  👍 0.0%
  Topic 44 [abandoned, unfinished, devs]
    Reviews: 28  |  👎 92.9%  |  👍 7.1%
  Topic 64 [idea, execution, demo]
    Reviews: 24  |  👎 87.5%  |  👍 12.5%
  Topic 74 [auto, basic, alot]
    Reviews: 21  |  👎 85.7%  |  👍 14.3%
  Topic 76 [talk, npc, tedious]
    Reviews: 20  |  👎 85.0%  |  👍 15.0%
  Topic 79 [fps, really recommend, gpu]
    Reviews: 20  |  👎 85.0%  |  👍 15.0%
  Topic 91 [gb, crashes, memory]
    Reviews: 18  |  👎 83.3%  |  👍 16.7%
  Topic 140 [patch, patches, early]
    Reviews: 12  |  👎 83.3%  |  👍 16.7%
  Topic 40 [fps, dlss, optimization]
    Reviews: 29  |  👎 82.8%  |  👍 17.2%
  Topic 155 [horrible, load, trouble]
    Reviews: 11  |  👎 81.8%  |  👍 18.2%
  Topic 114 [height, target, useful]
    Reviews: 15  |  👎 80.0%  |  👍 20.0%
  Topic 14 [buggy mess, buggy, breaking bugs]
    Reviews: 38  |  👎 76.3%  |  👍 23.7%
  Topic

### **Ninja Gaiden 4 w/ DLC**

In [ ]:
# Ninja Gaiden 4
# data = analyze_full_game(2627260)

# Ninja Gaiden 4 w/ DLC
data = analyze_full_game(2627260, dlc_app_id=4191490)


===== BASE GAME: NINJA GAIDEN 4 (ID: 2627260) =====

=== PLAYER SNAPSHOT: NINJA GAIDEN 4 ===
  Current Players:  316


Fetching base: 100%|██████████| 4000/4000 [01:06<00:00, 60.59it/s]
2026-05-08 14:50:01,493 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.71      0.90      0.80       121
    Recommend       0.98      0.94      0.96       679

     accuracy                           0.93       800
    macro avg       0.85      0.92      0.88       800
 weighted avg       0.94      0.93      0.93       800


Top words driving NOT RECOMMEND:
  boring                    coef: -3.5459
  just                      coef: -3.0454
  refund                    coef: -2.8812
  bad                       coef: -2.5208
  refunded                  coef: -2.1937
  terrible                  coef: -2.1275
  trash                     coef: -2.0910
  change                    coef: -2.0327
  maybe                     coef: -1.9859
  worse                     coef: -1.9836
  crash                     coef: -1.9677
  broken                    coef: -1.8230
  money                     coef: -1.7532
  worst                     coef: -1.7458
  repetitive 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 14:57:56,978 - BERTopic - Embedding - Completed ✓
2026-05-08 14:57:56,980 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:58:10,373 - BERTopic - Dimensionality - Completed ✓
2026-05-08 14:58:10,375 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 14:58:10,567 - BERTopic - Cluster - Completed ✓
2026-05-08 14:58:10,575 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 14:58:10,925 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                     Name  \
0       0     19  0_really good_disappointing_level_price   
1       1     19  1_playable_campaign_disappointing_ayane   
2       2     18                     2_going_ng_far_doesn   
3       3     18               3_maybe_budget_levels_didn   
4       4     18       4_honestly_scythe_yakumo_basically   
5       5     18                5_compared_big_people_lot   
6       6     18               6_cut_cut content_buy_make   
7       7     18               7_cool_sale_character_main   
8       8     18                 8_half_way_new enemy_cut   
9       9     18                          9_say_fun_boss_   
10     10     18  10_experience_weapons ryu_attacks_quite   
11     11     18                 11_people_pretty_way_fun   
12     12     18              12_single_ve_literally_main   
13     13     18           13_chapter_ll_extremely_review   
14     14     18         14_love_hope_ninja gaiden_gaiden   

    

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-05-08 14:59:33,441 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 15:00:00,942 - BERTopic - Dimensionality - Completed ✓
2026-05-08 15:00:00,944 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 15:00:01,060 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 0 [ng, ve, ng ng]
    Reviews: 36  |  👎 100.0%  |  👍 0.0%
  Topic 14 [series, new, ninja]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 23 [crash, controller, crashes]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 34 [ng, ng black, enemies]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 50 [peak, music story, lord]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 55 [ayane, meant, kasumi]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 70 [chapter, lock, targeting]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 71 [yes, steam deck, deck]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 86 [miss, dlc, fps]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 91 [speak, voice, english]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 113 [work, dlc, spoiler]
    Reviews: 18  |  👎 100.0%  |  👍 0.0%
  Topic 95 [ryu, generic, weapons]
    Reviews: 127  |  👎 71.7%  |  👍 28.3%

--- MOST POSITIVE ---
  Topic 1 [ps, lol, came

### **Marvel Rivals**

In [ ]:
# Marvel Rivals
data = analyze_full_game(2767030)


===== BASE GAME: Marvel Rivals (ID: 2767030) =====

=== PLAYER SNAPSHOT: Marvel Rivals ===
  Current Players:  50,529


Fetching base: 100%|██████████| 4000/4000 [00:59<00:00, 66.94it/s]
2026-05-08 15:01:01,827 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.93      0.86      0.89       609
    Recommend       0.63      0.79      0.70       191

     accuracy                           0.84       800
    macro avg       0.78      0.82      0.80       800
 weighted avg       0.86      0.84      0.85       800


Top words driving NOT RECOMMEND:
  eomm                      coef: -2.8808
  worse                     coef: -2.6725
  worst                     coef: -2.5197
  matchmaking               coef: -2.4833
  making                    coef: -2.2789
  anymore                   coef: -2.2727
  terrible                  coef: -2.0828
  just                      coef: -2.0730
  garbage                   coef: -2.0640
  trash                     coef: -1.9557
  season                    coef: -1.8327
  don                       coef: -1.8081
  unplayable                coef: -1.7780
  dont                      coef: -1.7600
  fix        

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 15:04:00,967 - BERTopic - Embedding - Completed ✓
2026-05-08 15:04:00,968 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 15:04:32,255 - BERTopic - Dimensionality - Completed ✓
2026-05-08 15:04:32,258 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 15:04:32,446 - BERTopic - Cluster - Completed ✓
2026-05-08 15:04:32,452 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 15:04:33,052 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                               Name  \
0      -1   1763                    -1_team_fun_players_matchmaking   
1       0    289                      0_eomm_win_matchmaking_ranked   
2       1    151             1_toxic_community_toxic community_chat   
3       2    131                        2_fun_shooter_es_characters   
4       3    119  3_matchmaking_match making_terrible_matchmakin...   
5       4    115                     4_bots_bot_matches_bot matches   
6       5    114                 5_marvel_marvel rivals_rivals_hero   
7       6    101           6_overwatch_better_better overwatch_love   
8       7     79         7_marvel_characters_love_marvel characters   
9       8     72                8_hate_life_addiction_mental health   
10      9     56                    9_matchmaking_win_matches_match   
11     10     55                  10_poke_buff_balancing_characters   
12     11     50                         11_healers_dps_he

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 15:07:28,420 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 15:07:28,435 - BERTopic - Dimensionality - Completed ✓
2026-05-08 15:07:28,436 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 15:07:28,573 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 20 [balancing, unbalanced, matchmaking]
    Reviews: 30  |  👎 100.0%  |  👍 0.0%
  Topic 22 [matchmaking, skill, team]
    Reviews: 30  |  👎 100.0%  |  👍 0.0%
  Topic 24 [ranked matchmaking, matchmaking, ranked]
    Reviews: 27  |  👎 100.0%  |  👍 0.0%
  Topic 25 [engagement, engagement based, based matchmaking]
    Reviews: 26  |  👎 100.0%  |  👍 0.0%
  Topic 45 [rigged, enemies, loss]
    Reviews: 14  |  👎 100.0%  |  👍 0.0%
  Topic 49 [run, optimize, fix ur]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 9 [matchmaking, win, matches]
    Reviews: 56  |  👎 98.2%  |  👍 1.8%
  Topic 12 [crashing, crashes, crash]
    Reviews: 48  |  👎 97.9%  |  👍 2.1%
  Topic 4 [bots, bot, matches]
    Reviews: 115  |  👎 97.4%  |  👍 2.6%
  Topic 29 [grandmaster, rank, gold]
    Reviews: 23  |  👎 95.7%  |  👍 4.3%
  Topic 0 [eomm, win, matchmaking]
    Reviews: 289  |  👎 94.1%  |  👍 5.9%
  Topic 21 [comp, qp, games]
    Reviews: 30  |  👎

### **HellDivers 2**

In [ ]:
# HellDivers 2
data = analyze_full_game(553850)


===== BASE GAME: HELLDIVERS™ 2 (ID: 553850) =====

=== PLAYER SNAPSHOT: HELLDIVERS™ 2 ===
  Current Players:  44,581


Fetching base: 100%|██████████| 4000/4000 [01:06<00:00, 60.08it/s]
2026-05-08 15:08:36,525 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.98      0.95      0.97       708
    Recommend       0.71      0.86      0.77        92

     accuracy                           0.94       800
    macro avg       0.84      0.91      0.87       800
 weighted avg       0.95      0.94      0.94       800


Top words driving NOT RECOMMEND:
  devs                      coef: -3.8175
  arrowhead                 coef: -3.3683
  fix                       coef: -3.2734
  balancing                 coef: -2.3390
  issues                    coef: -2.3361
  unplayable                coef: -2.1125
  balance                   coef: -2.0600
  ah                        coef: -1.9802
  performance               coef: -1.9247
  crashes                   coef: -1.8516
  warbond                   coef: -1.8339
  state                     coef: -1.8183
  make                      coef: -1.7604
  warbonds                  coef: -1.7574
  just       

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

2026-05-08 15:12:39,961 - BERTopic - Embedding - Completed ✓
2026-05-08 15:12:39,963 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 15:13:10,965 - BERTopic - Dimensionality - Completed ✓
2026-05-08 15:13:10,967 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-08 15:13:11,155 - BERTopic - Cluster - Completed ✓
2026-05-08 15:13:11,161 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-08 15:13:12,224 - BERTopic - Representation - Completed ✓



Top Topics Found:
    Topic  Count                                      Name  \
0      -1   1627                 -1_community_new_fun_devs   
1       0    639               0_pc_crashes_fix_unplayable   
2       1    331            1_arrowhead_community_fun_time   
3       2    259             2_helldivers_fun_time_players   
4       3    131      3_community_doxxed_charity_challenge   
5       4     89    4_credits_super credits_super_warbonds   
6       5     88  5_arrowhead_helldivers_players_community   
7       6     78          6_devs_hate_developers_community   
8       7     77     7_balance_balancing_balance team_team   
9       8     72                  8_fun_friends_best_games   
10      9     66                    9_devs_nerf_buff_nerfs   
11     10     65    10_democracy_super earth_earth_freedom   
12     11     56            11_weapons_enemies_weapon_want   
13     12     46                12_mechs_mech_buff_warbond   
14     13     44       13_performance_arrowhead_cra

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 15:17:13,647 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-08 15:17:13,662 - BERTopic - Dimensionality - Completed ✓
2026-05-08 15:17:13,663 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-08 15:17:13,803 - BERTopic - Cluster - Completed ✓



=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===

--- MOST NEGATIVE ---
  Topic 3 [community, doxxed, charity]
    Reviews: 131  |  👎 100.0%  |  👍 0.0%
  Topic 9 [devs, nerf, buff]
    Reviews: 66  |  👎 100.0%  |  👍 0.0%
  Topic 12 [mechs, mech, buff]
    Reviews: 46  |  👎 100.0%  |  👍 0.0%
  Topic 18 [coyote, nerf, nerfed]
    Reviews: 17  |  👎 100.0%  |  👍 0.0%
  Topic 22 [devs, changes, bugs]
    Reviews: 15  |  👎 100.0%  |  👍 0.0%
  Topic 23 [galactic, galactic war, war]
    Reviews: 14  |  👎 100.0%  |  👍 0.0%
  Topic 24 [devs, fired, refuse]
    Reviews: 13  |  👎 100.0%  |  👍 0.0%
  Topic 26 [pc, issue, delete]
    Reviews: 12  |  👎 100.0%  |  👍 0.0%
  Topic 27 [listen, devs, listen community]
    Reviews: 12  |  👎 100.0%  |  👍 0.0%
  Topic 28 [warbonds, weapons, arrowhead]
    Reviews: 11  |  👎 100.0%  |  👍 0.0%
  Topic 30 [arrowhead, hard, fix arrowhead]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 32 [devs, bugs, community]
    Reviews: 10  |  👎 100.0%  |  👍 0.0%
  Topic 13 [perform